# Hopfield networks: classic and modern associative memory

*ai · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/hopfield_memory)

A memory that completes a picture from a corrupted or half-hidden version. The classic Hopfield network stores binary patterns in Hebbian weights $W_{ij}$ and slides downhill in an energy; it holds at most about $0.138\,N$ random patterns, and far fewer correlated ones. The modern continuous Hopfield network of Ramsauer et al. (2020) keeps the patterns and updates $\xi \leftarrow X\,\mathrm{softmax}(\beta X^\top \xi)$, the attention of transformers: it retrieves cartoon faces in one step where the classic network mixes them up.

**Vector form**

$$W = \frac1N \sum_\mu \boldsymbol\xi^\mu \boldsymbol\xi^{\mu\top} - \tfrac PN I,\quad s_i \leftarrow \operatorname{sign}(W\mathbf s)_i,\quad E = -\tfrac12\,\mathbf s^\top W \mathbf s;\qquad \boldsymbol\xi \leftarrow X\,\operatorname{softmax}(\beta X^\top \boldsymbol\xi)$$

**Index (tensor) form**

$$W_{ij} = \frac1N\,\xi^\mu_i \xi^\mu_j\ (i \ne j),\qquad s_i \leftarrow \operatorname{sign}(W_{ij} s_j),\qquad p_\mu = \operatorname{softmax}_\mu(\beta\,\xi^\mu_n \xi_n),\qquad \xi_n \leftarrow p_\mu\,\xi^\mu_n$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\xi^\mu_i$ | `faces(P, 32, seed, device)` | (P, N) | pattern $\mu$: a procedurally drawn cartoon face, ink $+1$, paper $-1$, $N = 32 \times 32$ pixels |
| $W_{ij}$ | `einsum('fp,pi,pj->fij', used, X, X) / N` | (F, N, N) | Hebb's rule, repeated $\mu$ ⇒ summed; one network $f$ per number of stored faces, diagonal set to 0 |
| $h_i = W_{ij} s_j$ | `einsum('fqj,fj->fq', s, W[:, i])` | (F, Q) | repeated $j$ ⇒ summed: field on neuron $i$ for every network and query; $s_i$ takes its sign |
| $E = -\tfrac12 s_i W_{ij} s_j$ | `-0.5 * einsum('fqi,fij,fqj->fq', s, W, s)` | (F, Q) | energy; never rises under one-neuron-at-a-time updates |
| $p_\mu$ | `softmax(beta * einsum('qn,pn->qp', q, X), -1)` | (Q, P) | repeated $n$ ⇒ summed: similarity of the query to every stored pattern, sharpened by $\beta$ |
| $\xi_n \leftarrow p_\mu \xi^\mu_n$ | `einsum('qp,pn->qn', p, X)` | (Q, N) | repeated $\mu$ ⇒ summed: the new state is a weighted average of the patterns, one pattern when $\beta$ is large |
| $m = \tfrac1N s_i \xi_i$ | `overlap(s, target)` | (F, Q) | recall quality: 1 when the face is restored exactly |

The faces are not images of anyone: each one is drawn by the code from a few random numbers (head shape, eyes, glasses, brows, nose, mouth curvature, hair, ears), so no copyrighted characters or photographs are used. Each query hides the bottom 40 % of a face and flips 5 % of the other pixels. For random patterns the classic network's limit $\alpha_c \approx 0.138$ was computed by Amit, Gutfreund and Sompolinsky (1985) with methods from spin glasses, and the one-step bit error rate follows the Gaussian crosstalk law $\Phi(-1/\sqrt\alpha)$ (both checked in the tests). Faces are strongly correlated (they share an outline and a lot of empty paper), so the classic network already confuses them at a load far below $0.138$: it falls into mixtures of faces. The modern network's energy has a log-sum-exp term; Ramsauer et al. prove it decreases at every update, that one update usually suffices and that the capacity grows exponentially with $N$ for well separated patterns. Its update is exactly transformer attention with the stored patterns as keys and values. Make $\beta$ small to see metastable averages of several faces instead of one. The idea follows the blog post accompanying the paper, with procedural faces in place of its cartoon images.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/hopfield_memory/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import overlap

SIZE = 32  # faces are SIZE × SIZE pixels, so N = 1024 neurons


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_faces, beta = params["n_faces"], params["beta"]
    faces = tensors.faces(n_faces, SIZE, 0, dev)  # (P, S, S)
    queries = tensors.corrupt(faces, params["mask"], params["noise"], 1)  # (P, S, S)
    x, q = faces.reshape(n_faces, -1), queries.reshape(n_faces, -1)  # (P, N)
    n = x.shape[1]
    sims = overlap(x[:, None], x[None])  # (P, P)
    off = sims[~torch.eye(n_faces, dtype=torch.bool, device=dev)]
    print(f"{n_faces} faces drawn from random parameters, {SIZE}×{SIZE} = {n} neurons")
    print(f"faces are correlated: mean overlap between two faces {off.mean():.2f}, largest {off.max():.2f}")

    loads = torch.arange(1, n_faces + 1, device=dev)  # networks storing the first 1, 2, …, P faces
    used = (torch.arange(n_faces, device=dev)[None] < loads[:, None]).float()  # (F, P)
    classic = solve.classic_recall(x, used, q, params["sweeps"], 2)  # (F, P, N)
    modern = solve.modern_recall(x, used, q, beta)  # (F, P, N)
    m_classic = (overlap(classic, x[None]) * used).sum(-1) / used.sum(-1)  # mean over the stored faces (F,)
    m_modern = (overlap(modern, x[None]) * used).sum(-1) / used.sum(-1)
    for k in sorted({1, 2, 4, len(loads) // 2, len(loads) - 1}):
        print(
            f"{int(loads[k]):3d} faces stored (α = {loads[k] / n:.3f}): mean recall overlap "
            f"classic {m_classic[k]:.3f}   modern (β = {beta}) {m_modern[k]:.3f}"
        )

    alphas = torch.tensor([0.02, 0.05, 0.08, 0.1, 0.12, 0.14, 0.16, 0.2, 0.25, 0.3, 0.4, 0.6, 1.0, 1.5, 2.0])
    cap_c, cap_m, bit_err = solve.capacity_curve(alphas.to(dev), params["n_random"], 0.1, 8, params["sweeps"], 1.0, 3)
    print(f"random patterns, N = {params['n_random']}, 10 % of bits flipped:")
    for a, c, m, e in zip(alphas.tolist(), cap_c.tolist(), cap_m.tolist(), bit_err.tolist()):
        if round(a, 3) in (0.05, 0.1, 0.14, 0.2, 0.3, 1.0, 2.0):
            theory = 0.5 * math.erfc(1 / math.sqrt(2 * a))
            print(
                f"  α = {a:4.2f}: classic {c:.3f}  modern {m:.3f}   "
                f"one-step bit errors {e:.4f} (Φ(−1/√α) = {theory:.4f})"
            )
    print("the classic network fails past α ≈ 0.138 (and much earlier for correlated faces); the modern one does not")
    curve = (alphas, cap_c.cpu(), cap_m.cpu())
    return plot.animate(faces, queries, classic, modern, loads, m_classic, m_modern, curve, out_dir / "hopfield.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/hopfield_memory/equation.py
r"""Associative memory → tensor translation.

Classic Hopfield network (Hopfield 1982). N binary neurons s_i = ±1 store P patterns ξ^μ through Hebb's rule,

    W_{ij} = \frac1N \sum_\mu \xi^\mu_i \xi^\mu_j\ (i \ne j),\qquad W_{ii} = 0,

and recall by flipping one neuron at a time to the sign of its input field,

    s_i \leftarrow \operatorname{sign}\Big(\sum_j W_{ij} s_j\Big),\qquad E(s) = -\tfrac12\,s_i W_{ij} s_j.

With symmetric W and zero diagonal each single-neuron flip can only lower E, so the network slides into a local
minimum. For random patterns the stored ones stay (nearly) stable up to P ≈ 0.138 N (Amit, Gutfreund and
Sompolinsky 1985); beyond that, or with correlated patterns, the minima become mixtures and spurious states.

Modern continuous Hopfield network (Ramsauer et al. 2020, "Hopfield Networks is All You Need"). Keep the patterns
X = (ξ^1 … ξ^P) themselves instead of W and update a continuous state ξ by

    \xi \leftarrow X\,\operatorname{softmax}(\beta\,X^\top \xi),\qquad
    E(\xi) = -\beta^{-1}\log\sum_\mu e^{\beta\,\xi^{\mu\top}\xi} + \tfrac12\,\xi^\top\xi
             + \beta^{-1}\log P + \tfrac12 M^2,

the attention mechanism of transformers. Each update lowers E; for well separated patterns and large β one
update lands on the closest pattern, and the number of storable patterns grows exponentially with N.
"""

import torch


def hebbian(patterns: torch.Tensor, used: torch.Tensor) -> torch.Tensor:
    r"""W_{fij} = (1/N) Σ_μ m_{fμ} ξ^μ_i ξ^μ_j with W_{fii} = 0   (μ summed; m_{fμ} = 1 if pattern μ is stored in
    network f).   (P, N), (F, P) → (F, N, N)"""
    n = patterns.shape[-1]
    w = torch.einsum("fp,pi,pj->fij", used, patterns, patterns) / n
    return w * (1 - torch.eye(n, device=w.device))


def field(s: torch.Tensor, w_row: torch.Tensor) -> torch.Tensor:
    r"""h_i = Σ_j W_{ij} s_j for one neuron i in every network and query (j summed).   (F, Q, N), (F, N) → (F, Q)"""
    return torch.einsum("fqj,fj->fq", s, w_row)


def energy(s: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    r"""E = −½ s_i W_{ij} s_j   (i, j summed).   (F, Q, N), (F, N, N) → (F, Q)"""
    return -0.5 * torch.einsum("fqi,fij,fqj->fq", s, w, s)


def async_sweep(s: torch.Tensor, w: torch.Tensor, order: torch.Tensor) -> torch.Tensor:
    """One asynchronous sweep: neurons in the given order, each set to the sign of its field (kept if the field is
    0). All networks and queries update in parallel; only the neurons go one by one.   (F, Q, N) → (F, Q, N)"""
    s = s.clone()
    for i in order.tolist():
        h = field(s, w[:, i])  # (F, Q)
        s[..., i] = torch.where(h == 0, s[..., i], torch.sign(h))
    return s


def modern_update(xi: torch.Tensor, patterns: torch.Tensor, beta: float) -> torch.Tensor:
    r"""ξ ← X softmax(β Xᵀ ξ): attention of the query over the stored patterns.   (..., N), (P, N) → (..., N)"""
    p = torch.softmax(beta * torch.einsum("...n,pn->...p", xi, patterns), dim=-1)  # (..., P) weights of the patterns
    return torch.einsum("...p,pn->...n", p, patterns)


def modern_energy(xi: torch.Tensor, patterns: torch.Tensor, beta: float) -> torch.Tensor:
    r"""E = −β⁻¹ lse(β Xᵀξ) + ½ ξᵀξ + β⁻¹ log P + ½ M², M = max_μ |ξ^μ|.   (..., N), (P, N) → (...)"""
    lse = torch.logsumexp(beta * torch.einsum("...n,pn->...p", xi, patterns), dim=-1)
    m2 = patterns.pow(2).sum(-1).max()
    n_pat = torch.tensor(float(patterns.shape[0]))
    return -lse / beta + 0.5 * xi.pow(2).sum(-1) + torch.log(n_pat) / beta + 0.5 * m2


def overlap(s: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    r"""m = (1/N) Σ_i s_i ξ_i: 1 for perfect recall, about 0 for an unrelated state.   (..., N) → (...)"""
    return (s * target).sum(-1) / s.shape[-1]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/hopfield_memory/tensors.py
"""Build the concrete tensors: procedurally drawn cartoon faces, random patterns and corrupted queries.

The faces are drawn here from a handful of random numbers each (head shape, eyes, brows, mouth, hair, glasses,
ears) with distance fields on a pixel grid: no images or characters are copied from anywhere.
"""

import torch


def _grid(size: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Pixel centres y (down) and x (right) in [−1, 1]: two (size, size) tensors."""
    c = (torch.arange(size, device=device, dtype=torch.float32) + 0.5) / size * 2 - 1
    return torch.meshgrid(c, c, indexing="ij")


def faces(n: int, size: int, seed: int, device: torch.device) -> torch.Tensor:
    """n cartoon faces as ±1 images (ink +1, paper −1): shape (n, size, size). Every feature is broadcast over the
    batch, so all faces are drawn at once from their (n, 1, 1) parameters."""
    gen = torch.Generator().manual_seed(seed)

    def u(lo: float, hi: float) -> torch.Tensor:  # one uniform number per face, shaped (n, 1, 1) to broadcast
        return (lo + (hi - lo) * torch.rand(n, 1, 1, generator=gen)).to(device)

    y, x = _grid(size, device)  # (S, S)
    # head outline: an ellipse ring
    a, b = u(0.62, 0.86), u(0.72, 0.92)
    head = ((x / a) ** 2 + ((y - 0.05) / b) ** 2).sqrt()  # (n, S, S) 1 on the outline
    ink = (head - 1).abs() < 0.07
    # ears: small rings on both sides (half of the faces)
    ears = (u(0, 1) < 0.5) & ((((x.abs() - a) / 0.11) ** 2 + ((y - 0.05) / 0.17) ** 2).sqrt() - 1).abs().lt(0.35)
    ink |= ears & (head > 1)
    # hair: a filled cap above a random hairline, or nothing
    hairline = u(-0.75, -0.35)
    ink |= (u(0, 1) < 0.65) & (head < 1.06) & (y < hairline + 0.08 * torch.cos(6 * x * u(0.5, 1.5)))
    # eyes: two discs, with optional glasses around them
    ex, ey, er = u(0.22, 0.38), u(-0.18, 0.02), u(0.07, 0.13)
    eye = (((x.abs() - ex) ** 2 + (y - ey) ** 2).sqrt() < er) & (head < 1)
    glasses = (u(0, 1) < 0.35) & ((((x.abs() - ex) ** 2 + (y - ey) ** 2).sqrt() - 2 * er - 0.04).abs() < 0.04)
    ink |= eye | (glasses & (head < 1))
    # eyebrows: tilted bars above the eyes
    tilt = u(-0.5, 0.5)
    brow_y = ey - er - u(0.08, 0.16) + tilt * (x.abs() - ex)
    ink |= ((x.abs() - ex).abs() < 0.14) & ((y - brow_y).abs() < 0.035) & (head < 1)
    # nose: a short vertical stroke
    ink |= (x.abs() < 0.035) & (y > ey + 0.05) & (y < ey + u(0.18, 0.3))
    # mouth: a parabola y = m0 + c x² (c > 0 smiles, c < 0 frowns) of random width
    m0, c, mw = u(0.38, 0.55), u(-0.9, 1.6), u(0.18, 0.4)
    ink |= ((y - (m0 + c * (x**2 - mw**2))).abs() < 0.045) & (x.abs() < mw)
    return ink.float() * 2 - 1


def random_patterns(n: int, dim: int, seed: int, device: torch.device) -> torch.Tensor:
    """n random ±1 patterns of dimension dim, each bit a fair coin: (n, dim). The textbook capacity case."""
    gen = torch.Generator().manual_seed(seed)
    return (torch.randint(0, 2, (n, dim), generator=gen) * 2 - 1).float().to(device)


def corrupt(patterns: torch.Tensor, mask_rows: float, flip: float, seed: int) -> torch.Tensor:
    """Queries: blank (−1) the bottom fraction mask_rows of every image and flip a fraction `flip` of the other
    pixels.   (P, S, S) → (P, S, S)"""
    gen = torch.Generator().manual_seed(seed)
    q = patterns.clone()
    size = q.shape[-1]
    q[..., size - int(mask_rows * size) :, :] = -1
    flips = (torch.rand(q.shape, generator=gen) < flip).to(q.device)
    return torch.where(flips, -q, q)


def flip_bits(patterns: torch.Tensor, flip: float, seed: int) -> torch.Tensor:
    """Flip a random fraction `flip` of the bits of every pattern.   (P, N) → (P, N)"""
    gen = torch.Generator().manual_seed(seed)
    flips = (torch.rand(patterns.shape, generator=gen) < flip).to(patterns.device)
    return torch.where(flips, -patterns, patterns)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/hopfield_memory/solve.py
"""Recall with both networks: corrupted faces for a growing number of stored faces, and the capacity curves."""

import torch

from problems._common.progress import steps

from .equation import async_sweep, hebbian, modern_update, overlap


def classic_recall(
    patterns: torch.Tensor, used: torch.Tensor, queries: torch.Tensor, sweeps: int, seed: int
) -> torch.Tensor:
    """Run F classic networks (network f stores the patterns with used[f] = 1) from the same queries.
    (P, N), (F, P), (Q, N) → final states (F, Q, N)."""
    w = hebbian(patterns, used)  # (F, N, N)
    s = queries[None].expand(len(used), *queries.shape).clone()
    gen = torch.Generator().manual_seed(seed)
    for _ in steps(sweeps, "sweeps"):
        s = async_sweep(s, w, torch.randperm(patterns.shape[-1], generator=gen))
    return s


def modern_recall(patterns: torch.Tensor, used: torch.Tensor, queries: torch.Tensor, beta: float) -> torch.Tensor:
    """One modern update per network: only stored patterns take part (the others get −∞ similarity).
    (P, N), (F, P), (Q, N) → (F, Q, N)."""
    sim = beta * torch.einsum("qn,pn->qp", queries, patterns)  # (Q, P)
    sim = sim[None].masked_fill(used[:, None, :] == 0, float("-inf"))  # (F, Q, P)
    return torch.einsum("fqp,pn->fqn", torch.softmax(sim, dim=-1), patterns)


def capacity_curve(
    alphas: torch.Tensor, n: int, flip: float, n_queries: int, sweeps: int, beta: float, seed: int
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Mean recall overlap of random patterns against the load α = P/N, from queries with a fraction `flip` of bits
    flipped: classic (A,), modern (A,), and the one-step bit error rate of the stored patterns (A,)."""
    loads = (alphas * n).round().long().clamp(min=n_queries)  # P for every α
    patterns = torch.randint(0, 2, (int(loads.max()), n), generator=torch.Generator().manual_seed(seed))
    patterns = (patterns * 2 - 1).float().to(alphas.device)  # (P_max, N)
    used = (torch.arange(len(patterns), device=alphas.device)[None] < loads[:, None]).float()  # (A, P_max)
    targets = patterns[:n_queries]  # (Q, N), stored in every network
    gen = torch.Generator().manual_seed(seed + 1)
    flips = (torch.rand(targets.shape, generator=gen) < flip).to(targets.device)
    queries = torch.where(flips, -targets, targets)

    classic = overlap(classic_recall(patterns, used, queries, sweeps, seed + 2), targets).mean(-1)
    modern = overlap(modern_recall(patterns, used, queries, beta), targets).mean(-1)
    w = hebbian(patterns, used)  # (A, N, N)
    h = torch.einsum("fij,pj->fpi", w, patterns)  # (A, P_max, N) one synchronous step from every pattern
    wrong = (torch.sign(h) != patterns[None]).float().mean(-1)  # (A, P_max)
    bit_error = (wrong * used).sum(-1) / used.sum(-1)
    return classic, modern, bit_error


def mixture(patterns: torch.Tensor, queries: torch.Tensor, beta: float, n_steps: int) -> torch.Tensor:
    """n_steps modern updates of every query: (P, N), (Q, N) → (Q, N)."""
    xi = queries
    for _ in range(n_steps):
        xi = modern_update(xi, patterns, beta)
    return xi


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/hopfield_memory/plot.py
"""Corrupted faces recalled by both networks as more faces are stored, next to the capacity curves."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

SHOWN = 5  # faces shown in the image grid
ROWS = ["query", "classic", "modern", "stored"]


def animate(
    faces: torch.Tensor,
    queries: torch.Tensor,
    classic: torch.Tensor,
    modern: torch.Tensor,
    loads: torch.Tensor,
    m_classic: torch.Tensor,
    m_modern: torch.Tensor,
    curve: tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    path: Path,
) -> Path:
    """faces, queries (P, S, S); classic, modern (F, P, N); loads, m_classic, m_modern (F,); curve = α, classic,
    modern overlaps of random patterns."""
    s = faces.shape[-1]
    n = s * s
    img_c = classic[:, :SHOWN].reshape(len(loads), SHOWN, s, s).cpu().numpy()
    img_m = modern[:, :SHOWN].reshape(len(loads), SHOWN, s, s).cpu().numpy()
    fixed = {"query": queries[:SHOWN].cpu().numpy(), "stored": faces[:SHOWN].cpu().numpy()}
    p = loads.cpu().numpy()
    mc, mm = m_classic.cpu().numpy(), m_modern.cpu().numpy()

    fig = plt.figure(figsize=(10, 5.4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.15, 1], hspace=0.5, wspace=0.18)
    inner = grid[:, 0].subgridspec(len(ROWS), SHOWN, hspace=0.08, wspace=0.08)
    images = {}
    for r, row in enumerate(ROWS):
        for c in range(SHOWN):
            ax = fig.add_subplot(inner[r, c])
            data = fixed[row][c] if row in fixed else np.zeros((s, s))
            images[row, c] = ax.imshow(-data, cmap="gray", vmin=-1, vmax=1, interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
            ax.grid(False)
            if c == 0:
                ax.set_ylabel(row, fontsize=9)
    head = fig.text(0.29, 0.95, "", ha="center", fontsize=10)

    ax_f = fig.add_subplot(grid[0, 1])
    ax_f.plot(p, mc, color="#d1d5db", lw=1)
    ax_f.plot(p, mm, color="#d1d5db", lw=1)
    (line_c,) = ax_f.plot([], [], "o-", color="#dc2626", ms=3, lw=1.5, label="classic (Hebb)")
    (line_m,) = ax_f.plot([], [], "s-", color="#2563eb", ms=3, lw=1.5, label="modern (softmax)")
    ax_f.set_ylim(0.3, 1.04)
    ax_f.set_xlim(0, p[-1] + 1)
    ax_f.set_xlabel("faces stored P")
    ax_f.set_ylabel("recall overlap m")
    ax_f.set_title("corrupted faces", fontsize=10)
    ax_f.legend(fontsize=8, frameon=False, loc="lower left")

    alphas, cap_c, cap_m = (t.numpy() for t in curve)
    ax_a = fig.add_subplot(grid[1, 1])
    ax_a.semilogx(alphas, cap_c, "o-", color="#dc2626", ms=3, lw=1.5, label="classic")
    ax_a.semilogx(alphas, cap_m, "s-", color="#2563eb", ms=3, lw=1.5, label="modern")
    ax_a.axvline(0.138, color="#f59e0b", ls="--", lw=1.2, label="α_c ≈ 0.138")
    load = ax_a.axvline(p[0] / n, color="#6b7280", lw=1, ls=":", label="faces now")
    ax_a.set_ylim(0.3, 1.04)
    ax_a.set_xlim(8e-4, 2.5)
    ax_a.set_xlabel("load α = P / N")
    ax_a.set_ylabel("recall overlap m")
    ax_a.set_title("random ±1 patterns (10 % bits flipped)", fontsize=10)
    ax_a.legend(fontsize=7, frameon=False, loc="lower left")

    def update(k: int):
        for c in range(SHOWN):
            stored = c < p[k]  # faces not stored yet stay grey
            images["classic", c].set_data(-img_c[k, c] if stored else np.zeros((s, s)))
            images["modern", c].set_data(-img_m[k, c] if stored else np.zeros((s, s)))
        line_c.set_data(p[: k + 1], mc[: k + 1])
        line_m.set_data(p[: k + 1], mm[: k + 1])
        load.set_xdata([p[k] / n, p[k] / n])
        head.set_text(f"{p[k]} faces stored in N = {n} neurons (α = {p[k] / n:.3f})")

    return save_gif(fig, update, frame_indices(len(p), 90), path, fps=4)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_faces': 36,
    'beta': 0.1,
    'mask': 0.4,
    'noise': 0.05,
    'sweeps': 5,
    'n_random': 400
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.hopfield_memory")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.hopfield_memory.main")
problem_dir = Path("problems/ai/hopfield_memory")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
